# Molecular dynamics of two-dimensional argon

This notebook illustrates a small molecular-dynamics simulation and a few basic ideas of statistical physics. Argon atoms interact through a Lennard--Jones potential in a square box with periodic boundary conditions. SI units are used throughout.


### Compilation with Numba

The decorator `@njit` in some of the routined below asks Numba to translate a numerical Python function into machine code. Compilation occurs when the function is called for the first time with a particular combination of argument types; later calls reuse that compiled specialization. This requires to install `numba` into the Pythin environment.

Precompilation can considerably speed up the code, provided that it is written in a Numba compatible way. For efficient compilation, the kernel uses NumPy arrays with fixed numerical element types and simple loops and arithmetic. Arbitrary Python objects and containers with changing types are not suitable for `njit`. Constants such as the box size and cutoff are globals captured when the function is compiled. If those setup parameters are changed interactively, rerun the cells defining the compiled functions before starting a new simulation. Plotting, progress output, and file handling remain ordinary Python because compiling them provides no benefit.


In [ ]:
from pathlib import Path
import time

import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
from numba import njit
from IPython.display import Video, display

notebook_name = "MD_Argon"
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".git").exists()),
    Path.cwd(),
)
output_dir = project_root / "data" / "02_statphys" / notebook_name
output_dir.mkdir(parents=True, exist_ok=True)

# Compact layout used for figures in the lecture notes.
latex_textwidth_in = 6.3
single_panel = (0.62 * latex_textwidth_in, 0.42 * latex_textwidth_in)
full_width = (latex_textwidth_in, 0.56 * latex_textwidth_in)
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,
    "lines.linewidth": 1.2,
    "axes.linewidth": 0.8,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

rng = np.random.default_rng(2627)
print(f"Output directory: {output_dir}")

## Lennard--Jones interaction

For two atoms separated by $r$, the Lennard--Jones potential is

$$U(r)=4\epsilon\left[\left(\frac{\sigma}{r}\right)^{12}-\left(\frac{\sigma}{r}\right)^6\right].$$

We truncate it at $r_c=2.5\sigma$. A force shift makes both the potential and force continuous at the cutoff, which improves energy conservation.

In [ ]:
kB = 1.380649e-23  # J/K
nm = 1e-9              # m
fs = 1e-15             # s

# Lennard--Jones parameters (J. Chem. Phys. data cited in the lecture notes).
Ne_sigma, Ne_epsilon = 0.28000 * nm, 39.83 * kB
Ar_sigma, Ar_epsilon = 0.33916 * nm, 137.90 * kB
Kr_sigma, Kr_epsilon = 0.36233 * nm, 191.52 * kB
Ar_mass = 66.34e-27  # kg

cutoff_sigma = 2.5

@njit
def raw_lj_potential(r, sigma, epsilon):
    sr6 = (sigma / r) ** 6
    return 4 * epsilon * (sr6**2 - sr6)

@njit
def raw_lj_radial_force(r, sigma, epsilon):
    sr6 = (sigma / r) ** 6
    return 24 * epsilon * (2 * sr6**2 - sr6) / r

@njit
def lj_potential_scalar(r, sigma, epsilon):
    """Force-shifted Lennard--Jones potential for one distance."""
    rc = cutoff_sigma * sigma
    shifted = (
        raw_lj_potential(r, sigma, epsilon)
        - raw_lj_potential(rc, sigma, epsilon)
        + (r - rc) * raw_lj_radial_force(rc, sigma, epsilon)
    )
    return shifted if r < rc else 0.0

def lj_potential(r, sigma, epsilon):
    """Vectorized form used for plotting."""
    rc = cutoff_sigma * sigma
    shifted = (
        raw_lj_potential(r, sigma, epsilon)
        - raw_lj_potential(rc, sigma, epsilon)
        + (r - rc) * raw_lj_radial_force(rc, sigma, epsilon)
    )
    return np.where(r < rc, shifted, 0.0)

fig, ax = plt.subplots(figsize=single_panel)
rs = np.linspace(0.75 * Ne_sigma, 1.5 * nm, 1000)
for label, sigma, epsilon in [
    ("Neon", Ne_sigma, Ne_epsilon),
    ("Argon", Ar_sigma, Ar_epsilon),
    ("Krypton", Kr_sigma, Kr_epsilon),
]:
    ax.plot(rs / nm, lj_potential(rs, sigma, epsilon) / (100 * kB), label=label)
ax.set(xlabel=r"$r$ [nm]", ylabel=r"$U/(100 k_B)$ [K]", ylim=(-2, 0.5))
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(output_dir / "lennard_jones_potentials.pdf")
plt.show()

In [ ]:
# In the following, we use argon.
LJ_sigma = Ar_sigma
LJ_epsilon = Ar_epsilon
LJ_cutoff = cutoff_sigma * LJ_sigma
mass = Ar_mass
LJ_rmin = LJ_sigma * 2 ** (1 / 6)

print(f"r_min = {LJ_rmin / nm:.4f} nm")
print(f"U(r_min)/kB = {lj_potential(LJ_rmin, LJ_sigma, LJ_epsilon) / kB:.2f} K")

## Geometry and initial state

The atoms move in two dimensions. Periodic boundary conditions represent an extended system without walls. The optional smooth external potential can be used to demonstrate an inhomogeneous equilibrium density.

### Linked cell list for the force calculation

Calculating all particle pairs would require $N(N-1)/2$ distance tests at every time step. Here the square box is therefore divided into cells whose side length equals the interaction cutoff $r_c$. Each particle is assigned to one cell from its current position.

Cell membership is stored in two integer arrays. `head[cell]` gives the first particle in a cell and `next_particle[i]` gives the next particle in the same cell. The value $-1$ marks the end of a list. For example,

```text
head[5] = 20,  next_particle[20] = 12,
next_particle[12] = 7,  next_particle[7] = -1
```

means that cell 5 contains particles 20, 12, and 7. Inserting particle `i` requires only `next_particle[i] = head[cell]` followed by `head[cell] = i`. Thus the complete list is rebuilt in $O(N)$ operations at every time step without Python dictionaries or dynamically sized lists.

A particle can interact only with particles in its own cell or one of the eight neighboring cells: any more distant cell is at least one cutoff away. Neighbor-cell indices are wrapped across the box boundaries. For every candidate pair, the **minimum-image convention** selects the shortest periodic displacement before the distance is tested against $r_c$.

The implementation below visits ordered pairs: once as $(i,j)$ and once as $(j,i)$. This directly accumulates the appropriate force on each particle. Pair energy is therefore divided by two at the end to avoid double counting.

In [ ]:
# Moderate defaults for interactive testing.
N1 = 16
N = N1**2
mean_spacing_factor = 2
mean_spacing = mean_spacing_factor * LJ_cutoff
L = mean_spacing * N1

bin_size = LJ_cutoff
bin_count = int(round(L / bin_size))

def confining_potential(x, y, depth):
    return -0.25 * depth * (np.cos(2 * np.pi * x / L) - 1) * (np.cos(2 * np.pi * y / L) - 1)

def confining_force(positions, depth):
    x, y = positions.T
    prefactor = -0.5 * np.pi * depth / L
    fx = prefactor * np.sin(2 * np.pi * x / L) * (np.cos(2 * np.pi * y / L) - 1)
    fy = prefactor * np.sin(2 * np.pi * y / L) * (np.cos(2 * np.pi * x / L) - 1)
    return np.column_stack((fx, fy))

In [ ]:
def init_positions(grid_scale=1):
    positions = np.zeros((N, 2))
    random_offset = 0.2 * mean_spacing
    for i in range(N1):
        for j in range(N1):
            base = np.array([i + 0.5, j + 0.5]) * mean_spacing * grid_scale
            base += 0.5 * (L - N1 * mean_spacing * grid_scale)
            positions[i * N1 + j] = (base + rng.uniform(-random_offset, random_offset, 2)) % L
    return positions

def init_velocities(temperature, mode="cardinal"):
    if mode == "cardinal":
        # Exactly four possible vectors: east, west, north, and south.
        # Equal populations of all directions make the total momentum exactly zero.
        if N % 4 != 0:
            raise ValueError("Cardinal initialization requires N to be divisible by four.")
        directions = np.array([[1, 0], [-1, 0], [0, 1], [0, -1]], dtype=float)
        velocities = np.repeat(directions, N // 4, axis=0)
        rng.shuffle(velocities)
        speed = np.sqrt(2 * kB * temperature / mass)
        return speed * velocities
    if mode == "gaussian":
        velocities = rng.normal(0.0, np.sqrt(kB * temperature / mass), size=(N, 2))
        velocities -= velocities.mean(axis=0)
        current_ke = kinetic_energy_per_particle(velocities)
        velocities *= np.sqrt(kB * temperature / current_ke)
        return velocities
    raise ValueError(f"Unknown velocity initialization mode: {mode}")

positions = init_positions()
fig, ax = plt.subplots(figsize=(3.2, 3.2))
ax.scatter(positions[:, 0] / nm, positions[:, 1] / nm, s=10)
ax.set(xlabel=r"$x$ [nm]", ylabel=r"$y$ [nm]", xlim=(0, L / nm), ylim=(0, L / nm))
ax.set_aspect("equal")
fig.tight_layout()
fig.savefig(output_dir / "initial_positions.pdf")
plt.show()

## Velocity-Verlet integration

Velocity Verlet is time reversible and symplectic. In the absence of an external thermostat, the simulation samples a microcanonical (NVE) trajectory: total energy should remain approximately constant while kinetic and potential energy are exchanged.

In [ ]:
@njit
def lennard_jones_forces(positions):
    num_particles = positions.shape[0]
    forces = np.zeros_like(positions)
    interaction_energy = 0.0

    # Linked cell list. A value of -1 marks the end of a list.
    head = np.full(bin_count * bin_count, -1, dtype=np.int64)
    next_particle = np.full(num_particles, -1, dtype=np.int64)

    # Insert every particle at the beginning of its cell's list.
    for i in range(num_particles):
        cell_x = int(positions[i, 0] // bin_size) % bin_count
        cell_y = int(positions[i, 1] // bin_size) % bin_count
        cell = cell_x * bin_count + cell_y
        next_particle[i] = head[cell]
        head[cell] = i

    # Search the particle's own cell and its eight neighbors.
    for i in range(num_particles):
        cell_x = int(positions[i, 0] // bin_size) % bin_count
        cell_y = int(positions[i, 1] // bin_size) % bin_count

        for cell_dx in (-1, 0, 1):
            for cell_dy in (-1, 0, 1):
                neighbor_x = (cell_x + cell_dx) % bin_count
                neighbor_y = (cell_y + cell_dy) % bin_count
                neighbor = neighbor_x * bin_count + neighbor_y

                # Follow the linked list of particles in this neighbor cell.
                j = head[neighbor]
                while j != -1:
                    if i != j:
                        dx = positions[i, 0] - positions[j, 0]
                        dy = positions[i, 1] - positions[j, 1]
                        dx -= L * np.rint(dx / L)
                        dy -= L * np.rint(dy / L)
                        r = np.sqrt(dx**2 + dy**2)

                        if 0 < r < LJ_cutoff:
                            radial_force = (
                                raw_lj_radial_force(r, LJ_sigma, LJ_epsilon)
                                - raw_lj_radial_force(LJ_cutoff, LJ_sigma, LJ_epsilon)
                            )
                            forces[i, 0] += radial_force * dx / r
                            forces[i, 1] += radial_force * dy / r
                            interaction_energy += lj_potential_scalar(
                                r, LJ_sigma, LJ_epsilon
                            )
                    j = next_particle[j]

    # Ordered pairs were counted twice, once from each particle.
    return forces, 0.5 * interaction_energy / num_particles

def kinetic_energy_per_particle(velocities):
    return 0.5 * mass * np.sum(velocities**2) / N

def external_energy_per_particle(positions, depth):
    return np.mean(confining_potential(positions[:, 0], positions[:, 1], depth))

def md_simulation(num_steps, dt, temperature, grid_scale, potential_depth, velocity_mode="cardinal"):
    positions = init_positions(grid_scale)
    velocities = init_velocities(temperature, mode=velocity_mode)
    all_positions = np.empty((num_steps, N, 2))
    all_velocities = np.empty((num_steps, N, 2))
    kinetic = np.empty(num_steps)
    external = np.empty(num_steps)
    interaction = np.empty(num_steps)

    lj_forces, pair_energy = lennard_jones_forces(positions)
    forces = lj_forces + confining_force(positions, potential_depth)
    start = time.perf_counter()

    for step in range(num_steps):
        all_positions[step] = positions
        all_velocities[step] = velocities
        kinetic[step] = kinetic_energy_per_particle(velocities)
        external[step] = external_energy_per_particle(positions, potential_depth)
        interaction[step] = pair_energy

        positions = (positions + velocities * dt + 0.5 * forces * dt**2 / mass) % L
        lj_forces, pair_energy = lennard_jones_forces(positions)
        new_forces = lj_forces + confining_force(positions, potential_depth)
        velocities += 0.5 * (forces + new_forces) * dt / mass
        forces = new_forces

        if step % max(1, num_steps // 20) == 0 or step == num_steps - 1:
            print(f"\rStep {step + 1:>{len(str(num_steps))}}/{num_steps}", end="")

    print(f"  ({time.perf_counter() - start:.1f} s)")
    return all_positions, all_velocities, kinetic, external, interaction

In [ ]:
T_init = 150.0       # K
grid_scale = 0.8
potential_depth = 0.0  # J; try, for example, 500*kB
velocity_mode = "cardinal"  # four initial velocities: east, west, north, south
dt = 1.0 * fs
num_steps = 1000000     # about 20 s on a typical laptop; use >=100_000 for lecture-quality statistics

thermal_displacement = dt * np.sqrt(10 * kB * T_init / mass)
print(f"dt = {dt / fs:.1f} fs, diagnostic displacement/sigma = {thermal_displacement / LJ_sigma:.3g}")

all_rs, all_vs, all_ekin, all_epot, all_eint = md_simulation(
    num_steps, dt, T_init, grid_scale, potential_depth, velocity_mode
)
times = np.arange(num_steps) * dt
## time for equilibration, used below. Depends on initial conditions.
equil_step = num_steps // 5

# Store a subsampled trajectory and all energy traces in the standard output directory.
save_stride = 10
np.savez_compressed(
    output_dir / "md_trajectory.npz",
    time=times[::save_stride],
    positions=all_rs[::save_stride],
    velocities=all_vs[::save_stride],
    kinetic_energy=all_ekin,
    external_energy=all_epot,
    interaction_energy=all_eint,
    box_length=L,
    equilibration_step=equil_step,
    velocity_mode=velocity_mode,
)
print(f"Saved trajectory to {output_dir / 'md_trajectory.npz'}")

## Energy conservation

The three energy contributions are shown per particle. After the initial relaxation, the total energy should fluctuate only weakly around a constant value. A systematic drift indicates that the time step is too large.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(single_panel[0], 1.55 * single_panel[1]), sharex=True)
series = [all_ekin, all_epot, all_eint, all_ekin + all_epot + all_eint]
labels = [r"$E_{\rm kin}/k_B$ [K]", r"$E_{\rm ext}/k_B$ [K]", r"$E_{\rm int}/k_B$ [K]", r"$E_{\rm tot}/k_B$ [K]"]
colors = ["C0", "C1", "C3", "black"]
for ax, values, label, color in zip(axes, series, labels, colors):
    ax.plot(times / (1000 * fs), values / kB, color=color)
    ax.set_ylabel(label)
axes[-1].set_xlabel(r"$t$ [ps]")
fig.tight_layout()
fig.savefig(output_dir / "energy_conservation.pdf")
plt.show()

## Optional trajectory animation

Video encoding is disabled by default because it adds to the runtime and requires `ffmpeg`. Set `make_animation = True` to create an MP4 file in the notebook's output directory.

In [ ]:
make_animation = True
if make_animation:
    animation_stride = max(1, num_steps // 500)
    sparse_positions = all_rs[::animation_stride]
    highlighted_particle = min(50, N - 1)
    fig, ax = plt.subplots(figsize=(3.2, 3.2))
    particles = ax.scatter([], [], s=18)
    highlighted = ax.scatter([], [], s=45, color="C3")
    title = ax.set_title("")
    ax.set(xlabel=r"$x$ [nm]", ylabel=r"$y$ [nm]", xlim=(0, L / nm), ylim=(0, L / nm))
    ax.set_aspect("equal")

    def update(frame):
        particles.set_offsets(sparse_positions[frame] / nm)
        highlighted.set_offsets(sparse_positions[frame, highlighted_particle:highlighted_particle + 1] / nm)
        title.set_text(f"t = {frame * animation_stride * dt / (1000 * fs):.1f} ps")
        return particles, highlighted, title

    ani = animation.FuncAnimation(fig, update, frames=len(sparse_positions), interval=40, blit=True)
    video_file = output_dir / "particle_motion.mp4"
    ani.save(video_file, writer="ffmpeg", dpi=150)
    plt.close(fig)
    display(Video(video_file, embed=True, width=360))

## Maxwell--Boltzmann distribution

Initially, every atom has the same speed and moves in one of only four directions: north, east, south, or west. The four directions occur equally often, so the total momentum vanishes. Collisions redistribute these discrete velocities and produce the continuous equilibrium distribution. For a two-dimensional classical gas, the equilibrium speed density is

$$p(v)=\frac{m}{k_BT}v\exp\left(-\frac{mv^2}{2k_BT}\right).$$

The temperature follows from equipartition, $\langle E_{\rm kin}\rangle=k_BT$ per particle in two dimensions.

In [ ]:
equilibrium_ekin = np.mean(all_ekin[equil_step:])
T_final = equilibrium_ekin / ((1 - 1 / N) * kB)
speeds = np.linalg.norm(all_vs, axis=2)
max_speed = np.max(speeds[equil_step:])
v = np.linspace(0, max_speed, 300)
mb_distribution = mass * v / (kB * T_final) * np.exp(-mass * v**2 / (2 * kB * T_final))
speed_bins = np.linspace(0, max_speed, 25)

edges = np.linspace(0, num_steps, 6, dtype=int)
fig, axes = plt.subplots(1, 5, figsize=(latex_textwidth_in, 0.34 * latex_textwidth_in), sharex=True, sharey=True)
for m, ax in enumerate(axes):
    start_step, end_step = edges[m], edges[m + 1]
    sampled_speeds = speeds[start_step:end_step].ravel()
    ax.hist(sampled_speeds, bins=speed_bins, density=True, alpha=0.65, color="C0")
    ax.plot(v, mb_distribution, color="C3")
    ax.set_title(f"{start_step*dt/(1000*fs):.0f}--{end_step*dt/(1000*fs):.0f} ps")
    ax.set_xlabel(r"$v$ [m/s]")
axes[0].set_ylabel(r"$p(v)$")
axes[-1].plot([], [], color="C3", label=fr"$T={T_final:.1f}$ K")
axes[-1].legend(frameon=False)
fig.tight_layout()
fig.savefig(output_dir / "maxwell_boltzmann.pdf")
plt.show()

## Number fluctuations

We count particles in square subareas of area $A$. Their time-dependent density is $\rho_A=N_A/A$. The variance of $N_A$ quantifies equilibrium number fluctuations; samples before `equil_step` are excluded.

In [ ]:
x1 = 0.25 * L
y1 = 0.25 * L
a_values = np.array([0.1, 0.2, 0.3, 0.4, 0.5]) * L
densities = np.empty((len(a_values), num_steps))

x = all_rs[:, :, 0]
y = all_rs[:, :, 1]
for idx, a in enumerate(a_values):
    counts = np.sum((x >= x1) & (x < x1 + a) & (y >= y1) & (y < y1 + a), axis=1)
    densities[idx] = counts / (a / nm) ** 2

fig, ax = plt.subplots(figsize=single_panel)
for density, a in zip(densities, a_values):
    ax.plot(times[::10] / (1000 * fs), density[::10], label=fr"$a={a/nm:.1f}$ nm")
ax.axvspan(0, times[equil_step] / (1000 * fs), color="0.9", label="equilibration")
ax.set(xlabel=r"$t$ [ps]", ylabel=r"$\rho_A$ [nm$^{-2}$]")
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(output_dir / "density_fluctuations.pdf")
plt.show()

In [ ]:
mean_density = np.mean(densities[:, equil_step:], axis=1)
std_density = np.std(densities[:, equil_step:], axis=1)
areas = (a_values / nm) ** 2

fig, axes = plt.subplots(1, 2, figsize=full_width)
axes[0].plot(areas, mean_density * areas, "+-", label=r"$\langle N_A\rangle$")
axes[0].plot(areas, std_density**2 * areas**2, "o-", label=r"$\mathrm{Var}(N_A)$")
axes[0].set(xlabel=r"$A$ [nm$^2$]", ylabel="particle number")
axes[0].legend(frameon=False)
axes[1].plot(areas, std_density**2 / mean_density, "o-")
axes[1].set(xlabel=r"$A$ [nm$^2$]", ylabel=r"$\mathrm{Var}(\rho_A)/\langle\rho_A\rangle$")
fig.tight_layout()
fig.savefig(output_dir / "number_fluctuation_scaling.pdf")
plt.show()

## Time-averaged spatial density

The final plot estimates the equilibrium density field after equilibration. With no external potential it should be spatially uniform, apart from finite-sampling noise.

In [ ]:
spatial_bins = 20
sampled_positions = all_rs[equil_step:].reshape(-1, 2)
counts, x_edges, y_edges = np.histogram2d(
    sampled_positions[:, 0], sampled_positions[:, 1],
    bins=spatial_bins, range=((0, L), (0, L)),
)
n_frames = num_steps - equil_step
bin_area_nm2 = ((L / spatial_bins) / nm) ** 2
spatial_density = counts / (n_frames * bin_area_nm2)

fig, ax = plt.subplots(figsize=(3.6, 3.2))
mesh = ax.pcolormesh(x_edges / nm, y_edges / nm, spatial_density.T, shading="flat", cmap="Blues")
fig.colorbar(mesh, ax=ax, label=r"$\langle\rho\rangle$ [nm$^{-2}$]")
if potential_depth > 0:
    x_grid, y_grid = np.meshgrid(np.linspace(0, L, 256), np.linspace(0, L, 256))
    potential_grid = confining_potential(x_grid, y_grid, potential_depth)
    levels = np.linspace(-0.95 * potential_depth, -0.05 * potential_depth, 10)
    ax.contour(x_grid / nm, y_grid / nm, potential_grid, levels=levels, colors="0.6", linewidths=0.7)
ax.set(xlabel=r"$x$ [nm]", ylabel=r"$y$ [nm]")
ax.set_aspect("equal")
fig.tight_layout()
fig.savefig(output_dir / "spatial_density.pdf")
plt.show()